# Batch1-Coding Solution

# Question 1: Image Preprocessing for Inference (PyTorch)
# Write a function to load an image and preprocess it for inference.

In [1]:
from PIL import Image
from torchvision import transforms

def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    return transform(image).unsqueeze(0)  # Add batch dimension


| Code                                                             | Purpose                | Explanation                                                                                |
| ---------------------------------------------------------------- | ---------------------- | ------------------------------------------------------------------------------------------ |
| `from PIL import Image`                                          | Import PIL Library     | Used to open and work with image files.                                                    |
| `from torchvision import transforms`                             | Import Transforms      | Used for image preprocessing operations like resize, tensor conversion, and normalization. |
| `def preprocess_image(image_path):`                              | Function Definition    | Creates a function that accepts an image path as input.                                    |
| `Image.open(image_path)`                                         | Open Image             | Loads the image from the specified location.                                               |
| `.convert("RGB")`                                                | Convert to RGB         | Ensures the image has 3 color channels (Red, Green, Blue).                                 |
| `transforms.Compose([...])`                                      | Create Pipeline        | Combines multiple preprocessing steps into one sequence.                                   |
| `transforms.Resize((224,224))`                                   | Resize Image           | Changes image size to 224×224 pixels, required by ResNet18.                                |
| `transforms.ToTensor()`                                          | Convert to Tensor      | Converts image into PyTorch tensor format and scales pixel values from 0-255 to 0-1.       |
| `transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])` | Normalize Image        | Applies ImageNet mean and standard deviation values used during ResNet training.           |
| `transform(image)`                                               | Apply Transformations  | Executes all preprocessing steps on the image.                                             |
| `.unsqueeze(0)`                                                  | Add Batch Dimension    | Changes shape from `[3,224,224]` to `[1,3,224,224]`.                                       |
| `return transform(image).unsqueeze(0)`                           | Return Processed Image | Returns the final image tensor ready for prediction.                                       |


# Question 2: Predict on New Image with a Trained Model
# Perform prediction and get the class label.

In [ ]:
model.eval()
input_image = preprocess_image("test.jpg")
with torch.no_grad():
    output = model(input_image)
    predicted_class = output.argmax(1).item()
print("Predicted Class:", predicted_class)


| Code                                         | Purpose                      | Explanation                                                                                           |
| -------------------------------------------- | ---------------------------- | ----------------------------------------------------------------------------------------------------- |
| `model.eval()`                               | Evaluation Mode              | Sets the model to testing mode. Disables training-specific layers like Dropout and BatchNorm updates. |
| `input_image = preprocess_image("test.jpg")` | Load Test Image              | Opens and preprocesses the image using the function created in Question 1.                            |
| `with torch.no_grad():`                      | Disable Gradient Calculation | Saves memory and speeds up prediction because no training is happening.                               |
| `output = model(input_image)`                | Model Prediction             | Passes the image through the trained model and gets prediction scores.                                |
| `output.argmax(1)`                           | Find Highest Score           | Selects the class index with the highest prediction score.                                            |
| `.item()`                                    | Convert to Integer           | Converts PyTorch tensor value into a normal Python integer.                                           |
| `predicted_class = ...`                      | Store Prediction             | Saves the predicted class label number.                                                               |
| `print("Predicted Class:", predicted_class)` | Display Result               | Prints the predicted class.                                                                           |


| Step | Operation            | Example Output         |
| ---- | -------------------- | ---------------------- |
| 1    | Load image           | `test.jpg`             |
| 2    | Resize & preprocess  | Tensor `[1,3,224,224]` |
| 3    | Send to model        | `model(input_image)`   |
| 4    | Model outputs scores | `[0.10, 0.85, 0.05]`   |
| 5    | Find maximum score   | `0.85`                 |
| 6    | Get class index      | `1`                    |
| 7    | Print result         | `Predicted Class: 1`   |


# Question 3: Build a CNN to classify CIFAR-10 images (PyTorch)
# Create a CNN model that classifies images from the CIFAR-10 dataset with accuracy above 60%.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

# CNN model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64*8*8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

# Training (1 epoch shown for simplicity)
for epoch in range(1):
    for images, labels in trainloader:
        outputs = net(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


| Section         | Code                                          | Purpose               | Explanation                                           |
| --------------- | --------------------------------------------- | --------------------- | ----------------------------------------------------- |
| Import          | `import torch`                                | PyTorch Library       | Main deep learning framework.                         |
| Import          | `import torch.nn as nn`                       | Neural Network Module | Provides layers like Conv2D, Linear, ReLU.            |
| Import          | `import torch.optim as optim`                 | Optimizer Module      | Used to update model weights.                         |
| Import          | `import torchvision`                          | Dataset Utilities     | Provides CIFAR-10 dataset.                            |
| Import          | `import torchvision.transforms as transforms` | Preprocessing         | Used for image transformations.                       |
| Preprocessing   | `transforms.Compose()`                        | Combine Steps         | Groups multiple preprocessing operations.             |
| Preprocessing   | `transforms.ToTensor()`                       | Convert Image         | Converts image to PyTorch tensor.                     |
| Preprocessing   | `transforms.Normalize((0.5,), (0.5,))`        | Normalize Data        | Scales pixel values for better training.              |
| Dataset         | `CIFAR10(train=True)`                         | Training Data         | Loads 50,000 training images.                         |
| Dataset         | `CIFAR10(train=False)`                        | Test Data             | Loads 10,000 testing images.                          |
| DataLoader      | `batch_size=64`                               | Batch Processing      | Processes 64 images at a time.                        |
| DataLoader      | `shuffle=True`                                | Random Training       | Randomizes image order during training.               |
| CNN Layer       | `nn.Conv2d(3,32,3,padding=1)`                 | Feature Extraction    | Extracts basic image features.                        |
| Activation      | `nn.ReLU()`                                   | Non-Linearity         | Helps model learn complex patterns.                   |
| Pooling         | `nn.MaxPool2d(2)`                             | Reduce Size           | Reduces image dimensions by half.                     |
| CNN Layer       | `nn.Conv2d(32,64,3,padding=1)`                | Deep Features         | Learns more advanced features.                        |
| Activation      | `nn.ReLU()`                                   | Non-Linearity         | Improves learning capability.                         |
| Pooling         | `nn.MaxPool2d(2)`                             | Reduce Size           | Further reduces image size.                           |
| Fully Connected | `nn.Linear(64*8*8,128)`                       | Hidden Layer          | Converts extracted features into meaningful patterns. |
| Activation      | `nn.ReLU()`                                   | Activation            | Introduces non-linearity.                             |
| Output Layer    | `nn.Linear(128,10)`                           | Classification        | Predicts one of 10 CIFAR-10 classes.                  |
| Forward Pass    | `x=self.conv(x)`                              | Feature Extraction    | Sends image through convolution layers.               |
| Flatten         | `x.view(x.size(0),-1)`                        | Reshape Data          | Converts 3D feature maps into 1D vector.              |
| Prediction      | `return self.fc(x)`                           | Final Output          | Produces class scores.                                |
| Model Creation  | `net=Net()`                                   | Create CNN            | Instantiates CNN model.                               |
| Loss Function   | `nn.CrossEntropyLoss()`                       | Calculate Error       | Measures prediction accuracy.                         |
| Optimizer       | `optim.Adam(...,lr=0.001)`                    | Update Weights        | Optimizes model parameters.                           |
| Training        | `for epoch in range(1)`                       | Training Loop         | Runs one complete training cycle.                     |
| Prediction      | `outputs=net(images)`                         | Forward Pass          | Generates predictions.                                |
| Loss            | `loss=criterion(outputs,labels)`              | Error Calculation     | Compares prediction with actual label.                |
| Gradient Reset  | `optimizer.zero_grad()`                       | Clear Gradients       | Removes previous gradients.                           |
| Backpropagation | `loss.backward()`                             | Compute Gradients     | Calculates weight updates.                            |
| Weight Update   | `optimizer.step()`                            | Learn                 | Updates model weights.                                |


# Question 4: Identify Overfitting from Training Logs and Solve It
# Problem: You notice the training accuracy increases but validation accuracy stagnates. Modify the model using dropout and early stopping(use mnist dataset)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = to_categorical(y_train), to_categorical(y_test)

model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(patience=3, restore_best_weights=True)

model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=30, callbacks=[early_stop])


This code detects and reduces overfitting in an MNIST classification model using Dropout layers and EarlyStopping, improving the model's ability to generalize to unseen data.

======

Overfitting: When a model performs very well on training data but poorly on unseen data.

| Code                                                          | Purpose              | Explanation                                                               |
| ------------------------------------------------------------- | -------------------- | ------------------------------------------------------------------------- |
| `from tensorflow.keras.models import Sequential`              | Import Model         | Used to create a sequential neural network.                               |
| `from tensorflow.keras.layers import Dense, Flatten, Dropout` | Import Layers        | Used to build neural network layers.                                      |
| `from tensorflow.keras.callbacks import EarlyStopping`        | Import Callback      | Stops training automatically when validation performance stops improving. |
| `from tensorflow.keras.datasets import mnist`                 | Load Dataset         | Imports MNIST handwritten digit dataset.                                  |
| `from tensorflow.keras.utils import to_categorical`           | One-Hot Encoding     | Converts labels into categorical format.                                  |
| `mnist.load_data()`                                           | Load Data            | Loads training and testing datasets.                                      |
| `x_train / 255.0`                                             | Normalize Data       | Converts pixel values from 0-255 to 0-1.                                  |
| `to_categorical(y_train)`                                     | Convert Labels       | Converts labels into one-hot encoded vectors.                             |
| `Sequential([...])`                                           | Create Model         | Defines neural network architecture.                                      |
| `Flatten(input_shape=(28,28))`                                | Flatten Layer        | Converts 28×28 image into 784 features.                                   |
| `Dense(256, activation='relu')`                               | Hidden Layer 1       | Learns complex patterns from data.                                        |
| `Dropout(0.3)`                                                | Regularization       | Randomly disables 30% neurons to reduce overfitting.                      |
| `Dense(128, activation='relu')`                               | Hidden Layer 2       | Learns deeper features.                                                   |
| `Dropout(0.3)`                                                | Regularization       | Further reduces overfitting.                                              |
| `Dense(10, activation='softmax')`                             | Output Layer         | Predicts one of 10 digit classes (0-9).                                   |
| `model.compile()`                                             | Configure Model      | Sets optimizer, loss function, and metric.                                |
| `optimizer='adam'`                                            | Optimizer            | Updates weights efficiently.                                              |
| `loss='categorical_crossentropy'`                             | Loss Function        | Measures prediction error for multi-class classification.                 |
| `metrics=['accuracy']`                                        | Evaluation Metric    | Tracks model accuracy.                                                    |
| `EarlyStopping(patience=3)`                                   | Early Stopping       | Stops training if validation performance doesn't improve for 3 epochs.    |
| `restore_best_weights=True`                                   | Best Model Recovery  | Restores weights from best-performing epoch.                              |
| `model.fit()`                                                 | Train Model          | Starts model training.                                                    |
| `validation_data=(x_test,y_test)`                             | Validation Data      | Monitors performance on unseen data.                                      |
| `epochs=30`                                                   | Maximum Epochs       | Allows training up to 30 epochs.                                          |
| `callbacks=[early_stop]`                                      | Apply Early Stopping | Enables automatic stopping.                                               |


# Question 5: Transfer Learning with Pretrained VGG16 (Cats vs Dogs)
# Problem: Use VGG16 for binary classification with fine-tuning

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout

base_model = VGG16(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = Flatten()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


It uses Transfer Learning with a pretrained VGG16 model and adds custom layers to classify images as either Cat or Dog.



| Code                                                          | Purpose                      | Explanation                                                |
| ------------------------------------------------------------- | ---------------------------- | ---------------------------------------------------------- |
| `from tensorflow.keras.applications import VGG16`             | Import VGG16                 | Loads the pretrained VGG16 model.                          |
| `from tensorflow.keras.models import Model`                   | Import Model                 | Used to create a custom model architecture.                |
| `from tensorflow.keras.layers import Dense, Flatten, Dropout` | Import Layers                | Used to build additional classification layers.            |
| `base_model = VGG16(...)`                                     | Load VGG16                   | Loads pretrained VGG16 with ImageNet weights.              |
| `include_top=False`                                           | Remove Original Output Layer | Excludes VGG16's 1000-class classifier.                    |
| `input_shape=(224,224,3)`                                     | Input Size                   | Defines RGB image size expected by VGG16.                  |
| `weights='imagenet'`                                          | Pretrained Weights           | Uses knowledge learned from ImageNet dataset.              |
| `for layer in base_model.layers:`                             | Access Layers                | Iterates through all VGG16 layers.                         |
| `layer.trainable=False`                                       | Freeze Layers                | Prevents pretrained weights from updating during training. |
| `x = base_model.output`                                       | Feature Extraction Output    | Gets extracted features from VGG16.                        |
| `Flatten()`                                                   | Flatten Layer                | Converts feature maps into a 1D vector.                    |
| `Dropout(0.5)`                                                | Reduce Overfitting           | Randomly disables 50% neurons during training.             |
| `Dense(128, activation='relu')`                               | Hidden Layer                 | Learns Cats vs Dogs specific features.                     |
| `Dense(1, activation='sigmoid')`                              | Output Layer                 | Produces binary classification output.                     |
| `Model(inputs=..., outputs=...)`                              | Create Final Model           | Combines VGG16 and custom layers.                          |
| `optimizer='adam'`                                            | Optimizer                    | Updates model weights efficiently.                         |
| `loss='binary_crossentropy'`                                  | Loss Function                | Calculates error for binary classification.                |
| `metrics=['accuracy']`                                        | Evaluation Metric            | Measures prediction accuracy.                              |
